In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
import time
from sklearn.preprocessing import MinMaxScaler
import warnings

# Suppress all UserWarnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)


traning_loss = []
accuracy = []


Using device: cpu


Generic Traing model

In [ ]:

def train_model(model, train_loader, val_loader, epochs=5, lr=0.01, name="Model"):
    """
    Generic training loop for regression tasks with validation.
    """
    # Move model to device (GPU or CPU)
    model = model.to(device)

    # Loss function for regression
    criterion = nn.MSELoss()

    # Optimizer (SGD with momentum)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    print(f"\nTraining {name} for {epochs} epochs...")
    start_time = time.time()

    final_val_loss = 0.0


    for epoch in range(epochs):
       # --- Training Phase ---
        model.train()
        running_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs.squeeze(), y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        # --- Validation Phase ---
        model.eval()
        val_loss = 0.0
        total = 0.0
        correct = 0.0
        mae = 0.0


        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                outputs = model(x)
                loss = criterion(outputs.squeeze(), y)
                val_loss += loss.item()
                mae += torch.mean(torch.abs(outputs - y)).item()
                # total += y.numel()
                # correct += ((torch.abs(outputs - y) / y) < 0.01).sum().item()
                epsilon = 1e-8
                correct += ((torch.abs(outputs.squeeze() - y) / (y + epsilon)) < 0.1).sum().item()
                total += y.numel()


        final_val_loss = val_loss / len(val_loader)
        final_val_acc = 100 * correct / total
        final_mae = mae / len(val_loader)

        print(f"  Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {final_val_loss:.4f} | Val Acc: {final_val_acc:.2f}% | MEA: {final_mae:.2f}")

    end_time = time.time()
    duration = end_time - start_time
    print(f"{name} - Final Accuracy: {final_val_acc:.2f}%, Time: {duration:.2f}s")

    return final_val_acc, final_val_loss, duration

Data preparation

In [ ]:
# Load the dataset
dataset = load_dataset("Adilbai/stock-dataset")
df = dataset["train"].to_pandas()

# Basic info
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Unique tickers: {df['Ticker'].nunique()}")

Dataset shape: (620095, 73)
Date range: 2020-07-15 00:00:00-04:00 to 2025-06-27 00:00:00-04:00
Unique tickers: 503


In [ ]:
# Sort by date for each ticker
df = df.sort_values(['Ticker', 'Date'])

# Only use numerical columns for prediction
features = ['Open', 'High', 'Low', 'Close', 'Volume']

ticker = df['Ticker'].unique()[0]
df_ticker = df[df['Ticker'] == ticker].reset_index(drop=True)

In [ ]:
class StockDataset(Dataset):
    def __init__(self, df, seq_len=10):
        self.df = df.reset_index(drop=True)
        self.seq_len = seq_len
        self.features = ['Open', 'High', 'Low', 'Close', 'Volume']

    def __len__(self):
        return len(self.df) - self.seq_len - 1

    def __getitem__(self, idx):
        x = self.df[self.features].iloc[idx:idx+self.seq_len].values

        current_close = self.df['Close'].iloc[idx+self.seq_len-1]
        next_close    = self.df['Close'].iloc[idx+self.seq_len]

        y = (next_close - current_close) / (current_close + 1e-8)
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32)
        )

In [ ]:
total_size = len(df_ticker)
train_size = int(0.7 * total_size)
val_size   = int(0.15 * total_size)
test_size  = total_size - train_size - val_size

train_df = df_ticker.iloc[:train_size].copy()
val_df   = df_ticker.iloc[train_size:train_size+val_size].copy()
test_df  = df_ticker.iloc[train_size+val_size:].copy()

scaler = MinMaxScaler()

train_df[features] = scaler.fit_transform(train_df[features])
val_df[features]   = scaler.transform(val_df[features])
test_df[features]  = scaler.transform(test_df[features])


seq_len = 25

train_dataset = StockDataset(train_df, seq_len)
val_dataset   = StockDataset(val_df, seq_len)
test_dataset  = StockDataset(test_df, seq_len)


train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f" Test: {len(test_dataset)}, Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# print(f"Test: {len(test_set)}, Train: {len(train_set)}, Val: {len(val_set)}")

 Test: 156, Train: 818, Val: 155


In [ ]:
# store the result
results = {}

Simple 1D CNN for practices

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, seq_len=25, n_features=5, n_classes=1):
        super(SimpleCNN, self).__init__()

        # Block 1
        self.conv1 = nn.Conv1d(in_channels=n_features, out_channels=32, kernel_size=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(2)

        # Block 2
        self.conv2 = nn.Conv1d(32, 64, kernel_size=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(2)

        # Block 3
        self.conv3 = nn.Conv1d(64, 128, kernel_size=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool1d(2)

        # Compute final sequence length after pooling
        self.final_seq_len = seq_len
        for _ in range(3):
            self.final_seq_len = self.final_seq_len // 2  # integer division

        # Classifier
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * self.final_seq_len, 512)
        self.relu_fc = nn.ReLU()
        self.drop = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, n_classes)  # 1 for regression, >1 for classification

    def forward(self, x):
        # x shape: [batch, seq_len, features], permute to [batch, features, seq_len]
        x = x.permute(0, 2, 1)

        x = self.pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu2(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu3(self.bn3(self.conv3(x))))

        x = self.flatten(x)
        x = self.relu_fc(self.fc1(x))
        x = self.drop(x)
        x = self.fc2(x)
        return x

# Initialize and train our complex CNN
my_cnn = SimpleCNN()
params_cnn = sum(p.numel() for p in my_cnn.parameters())
print(f"SimpleCNN Total Parameters: {params_cnn:,}")
acc, loss, t = train_model(my_cnn, train_loader, val_loader, epochs=10, name="Simple CNN")
results['Simple CNN'] = {'acc': acc, 'loss': loss, 'time': t, 'params': params_cnn}

SimpleCNN Total Parameters: 208,705

Training Simple CNN for 10 epochs...
  Epoch [1/10] | Train Loss: 0.0488 | Val Loss: 0.0027 | Val Acc: 50.97% | MEA: 0.04
  Epoch [2/10] | Train Loss: 0.0358 | Val Loss: 0.0036 | Val Acc: 49.68% | MEA: 0.05
  Epoch [3/10] | Train Loss: 0.0587 | Val Loss: 0.0210 | Val Acc: 45.81% | MEA: 0.13
  Epoch [4/10] | Train Loss: 0.0513 | Val Loss: 0.0167 | Val Acc: 48.39% | MEA: 0.12
  Epoch [5/10] | Train Loss: 0.0155 | Val Loss: 0.0061 | Val Acc: 47.10% | MEA: 0.06
  Epoch [6/10] | Train Loss: 0.0096 | Val Loss: 0.0064 | Val Acc: 45.81% | MEA: 0.06
  Epoch [7/10] | Train Loss: 0.0089 | Val Loss: 0.0035 | Val Acc: 47.74% | MEA: 0.04
  Epoch [8/10] | Train Loss: 0.0065 | Val Loss: 0.0037 | Val Acc: 51.61% | MEA: 0.04
  Epoch [9/10] | Train Loss: 0.0057 | Val Loss: 0.0033 | Val Acc: 46.45% | MEA: 0.04
  Epoch [10/10] | Train Loss: 0.0055 | Val Loss: 0.0030 | Val Acc: 45.81% | MEA: 0.04
Simple CNN - Final Accuracy: 45.81%, Time: 7.43s
